# 03 · Выравнивание на предпочтениях

**Цель:** та же, что в `01-sft` — мат о дипломе и только о нём, — но через пары «лучше / хуже» вместо готовых ответов. Из одной строки `swear.jsonl` пара получается перестановкой: на `topic` выбранный ответ с матом, отвергнутый обычный; на `other` наоборот. Метрика та же, цифры сравнимы с SFT напрямую.

Один ноутбук на четыре метода: `METHOD` переключает ORPO, DPO, SimPO, KTO. Математика — `books/03-alignment.pdf`.

In [ ]:
from common import MODEL_ID, SYSTEM, RUNS, read_raw, swears, demo_answers, show, side_by_side, swear_suite, fmt

import torch
from datasets import Dataset
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig
from vlmkit import Sample, memory_report, evaluate as ev
from vlmkit.compat import alignment_trainer, available_alignment, supported

print("доступно в вашем trl:", available_alignment())

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
suite = swear_suite()
before = demo_answers(model, processor)
before_metrics = ev.run(model, processor, suite)
show(before, "ДО ВЫРАВНИВАНИЯ", detector=swears)
print("\nметрики:", fmt(before_metrics))

# Отложенные пары для preference accuracy: целевой ответ против другого.
held = read_raw("swear.jsonl")[1::2]
good = [Sample.from_qa(r["question"], r["swear"] if r["group"] == "topic" else r["plain"]) for r in held]
bad  = [Sample.from_qa(r["question"], r["plain"] if r["group"] == "topic" else r["swear"]) for r in held]
pref_before = ev.preference_accuracy(model, processor, good, bad, system=SYSTEM)
print("preference accuracy:", pref_before)

## Пары

Тренеру пары отдаются в диалоговом формате: `prompt` — реплики с системным промптом, `chosen` и `rejected` — реплика ассистента. Так trl сам прогоняет их через chat template модели, и обучение видит тот же текст, что инференс. Сырые строки он взял бы как есть — без разметки ролей и без системного промпта.

In [ ]:
rows = read_raw("swear.jsonl")[::2]

def pair(r):
    good, bad = (r["swear"], r["plain"]) if r["group"] == "topic" else (r["plain"], r["swear"])
    return {
        "prompt":   [{"role": "system", "content": SYSTEM}, {"role": "user", "content": r["question"]}],
        "chosen":   [{"role": "assistant", "content": good}],
        "rejected": [{"role": "assistant", "content": bad}],
    }

pairs = Dataset.from_list([pair(r) for r in rows])
print(pairs)
p = pairs[0]
print(f"\nЗАПРОС:      {p['prompt'][1]['content']}")
print(f"ВЫБРАННЫЙ:   {p['chosen'][0]['content'][:160]}")
print(f"ОТВЕРГНУТЫЙ: {p['rejected'][0]['content'][:160]}")

## Тренер

`alignment_trainer` находит, где в вашей версии trl лежит класс под метод: в trl 1.x `ORPOTrainer` и `CPOTrainer` вынесены в `trl.experimental`, `KTOTrainer` остался на верхнем уровне, отдельного SimPO нет — это `CPOTrainer` с `loss_type="simpo"` и `cpo_alpha=0`. Обязательные для метода аргументы приходят в `extra`. Сам вызов ниже обычный.

Тренеру передаётся токенизатор, а не процессор: пары текстовые, chat template у них общий, а `ORPOTrainer` и `CPOTrainer` берут `pad_token_id` напрямую у `processing_class`.

KTO работает не с парами, а с отдельными ответами и меткой «хороший / плохой»; пары для него расшиваются на две строки.

Скорость обучения на порядок ниже, чем в SFT: выравнивание правит уже обученное поведение, и большой шаг его разрушает.

`beta` означает разное: у DPO — сила KL-штрафа (0.1), у ORPO — вес слагаемого с отношением шансов (0.1), у SimPO — масштаб награды (2.0–2.5). Меняйте вместе с методом.

In [ ]:
METHOD = "orpo"          # или simpo / dpo / kto — см. available_alignment() выше
Config, TrainerCls, extra = alignment_trainer(METHOD)
print(f"{METHOD} → {TrainerCls.__module__}.{TrainerCls.__name__} {extra}")

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True, bias="none", task_type="CAUSAL_LM",
)

if METHOD == "kto":      # не пары, а отдельные ответы с меткой
    train_data = Dataset.from_list(
        [{"prompt": r["prompt"], "completion": r["chosen"],   "label": True}  for r in pairs]
        + [{"prompt": r["prompt"], "completion": r["rejected"], "label": False} for r in pairs]
    )
    processor.tokenizer.padding_side = "left"   # требование KTOTrainer
else:
    train_data = pairs

args = Config(**supported(Config, {
    **extra,
    "output_dir": str(RUNS / METHOD),
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 3,
    "learning_rate": 5e-6,
    "beta": 2.0 if METHOD == "simpo" else 0.1,
    "bf16": True,
    "max_length": 1024,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "logging_steps": 5,
    "save_strategy": "no",
    "report_to": [],
    "remove_unused_columns": False,
}))

trainer = TrainerCls(
    model=model, args=args, train_dataset=train_data,
    processing_class=processor.tokenizer,
    peft_config=lora,       # выравнивание поверх адаптера: результат отключаемый
)
trainer.train()
model = trainer.model

## После

In [ ]:
model.eval()
after = demo_answers(model, processor)
after_metrics = ev.run(model, processor, suite)

show(after, f"ПОСЛЕ {METHOD.upper()}", detector=swears)
side_by_side(before, after, detector=swears)
print(f"\nдо:    {fmt(before_metrics)}")
print(f"после: {fmt(after_metrics)}")

pref_after = ev.preference_accuracy(model, processor, good, bad, system=SYSTEM)
print(f"preference accuracy: {pref_before['accuracy']:.0%} → {pref_after['accuracy']:.0%}, "
      f"зазор на токен {pref_before['margin']:+.3f} → {pref_after['margin']:+.3f}")

model.save_pretrained(str(RUNS / METHOD))

## Сравнить с SFT

Тот же замер, что в `01-sft`. Если recall такой же при меньшем FPR — пары научили границе лучше, чем примеры: отвергнутый ответ явно показывает, чего не делать. Если recall ниже — на 44 парах это нормально: выравнивание правит сложившееся поведение, а не создаёт новое, и связка SFT → DPO обычно сильнее любого из них по отдельности.